In [2]:
%load_ext autoreload
%autoreload 2
import sys; sys.path.append(r"C:\Users\dillo\OneDrive\Documents\Projects\Microsoft Mind")
import numpy as np, pandas as pd
from collections import Counter
from scipy import stats
from src.db import get_engine
from src.eval import ndcg_at_k, mrr
from src.metrics import intra_list_diversity
from src.recommenders import content_recommender
from src.rerank import make_reranker, evaluate_ranker
engine = get_engine()

emb = np.load("../models/news_emb.npy")
news_ids = np.load("../models/news_emb_ids.npy", allow_pickle=True)
row_of = {n: i for i, n in enumerate(news_ids)}

train_clicks = pd.read_sql("SELECT news_id, SUM(clicked) AS clicks FROM fact_impressions GROUP BY news_id",
                           engine).set_index("news_id")["clicks"].to_dict()
meta = pd.read_sql("SELECT news_id, category, subcategory FROM dim_news", engine)
cat_of    = dict(zip(meta.news_id, meta.category))
subcat_of = dict(zip(meta.news_id, meta.subcategory))

def parse(s): return [(nid, int(l)) for nid, l in (x.split('-') for x in s.split())]
def profile_from_history(h):
    if not isinstance(h, str) or not h.strip(): return None
    c, s = Counter(), Counter()
    for nid in h.split():
        if nid in cat_of:    c[cat_of[nid]] += 1
        if nid in subcat_of: s[subcat_of[nid]] += 1
    t = sum(c.values())
    return {"cats": c, "subs": s, "total": t} if t else None

dev = pd.read_csv("../data/raw/MINDsmall_dev/behaviors.tsv", sep="\t",
       names=["impression_id","user_id","time","history","impressions"])
dev["parsed"]  = dev["impressions"].apply(parse)
dev["profile"] = dev["history"].apply(profile_from_history)

base_score     = content_recommender(cat_of, subcat_of, train_clicks)
control_ranker = make_reranker(base_score, emb, row_of, lam=0.0)
treat_ranker   = make_reranker(base_score, emb, row_of, lam=0.5)

In [3]:
rng = np.random.default_rng(42)
users = dev["user_id"].unique()
sample_users = rng.choice(users, size=8000, replace=False)
arm = {u: ("treatment" if rng.random() < 0.5 else "control") for u in sample_users}

exp = dev[dev["user_id"].isin(sample_users)].copy()
exp["arm"] = exp["user_id"].map(arm)

def run_experiment(exp, k=10):
    recs = []
    for row in exp.itertuples(index=False):
        parsed = row.parsed
        if sum(l for _, l in parsed) == 0:
            continue
        ranker = treat_ranker if row.arm == "treatment" else control_ranker
        order  = ranker(row)
        lab    = dict(parsed)
        labels = [lab[nid] for nid in order]
        recs.append({"user_id": row.user_id, "arm": row.arm,
                     "ndcg": ndcg_at_k(labels, k), "mrr": mrr(labels),
                     "diversity": intra_list_diversity(order[:k], emb, row_of)})
    return pd.DataFrame(recs)

results = run_experiment(exp)
print(results.groupby("arm")[["ndcg", "mrr", "diversity"]].mean())

               ndcg       mrr  diversity
arm                                     
control    0.362962  0.324499   0.918092
treatment  0.357737  0.319998   0.935472


In [5]:
per_user = results.groupby(["arm", "user_id"])[["ndcg", "diversity"]].mean().reset_index()
for metric in ["ndcg", "diversity"]:
    c = per_user.loc[per_user.arm == "control",   metric].dropna()
    t = per_user.loc[per_user.arm == "treatment", metric].dropna()
    diff = t.mean() - c.mean()
    stat, p = stats.ttest_ind(t, c, equal_var=False)
    se = np.sqrt(t.var()/len(t) + c.var()/len(c))
    print(f"{metric:10s} lift={diff:+.4f}  95% CI=({diff-1.96*se:+.4f},{diff+1.96*se:+.4f})  p={p:.2e}")

ndcg       lift=-0.0008  95% CI=(-0.0145,+0.0129)  p=9.05e-01
diversity  lift=+0.0180  95% CI=(+0.0162,+0.0198)  p=2.15e-84
